# Maize Disease Classification
ResNet50 transfer learning on PlantVillage maize classes.

**Paths:** Notebook lives in `notebooks/`, so all project dirs use `../`
- `../data/` — datasets
- `../models/` — saved models
- `../logs/` — training logs

## Step 1 — Imports

In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import tensorflow as tf
from tensorflow.keras import Model, layers, Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger, TensorBoard
)
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import confusion_matrix, classification_report

print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Step 2 — Create Maize-Only Dataset
PlantVillage contains many crops. Run this **once** to extract only maize/corn classes.

In [ ]:
source_dir = '../data/plantvillage/color'  # update if your path differs

all_folders = os.listdir(source_dir)
maize_folders = [f for f in all_folders if 'corn' in f.lower() or 'maize' in f.lower()]

print(f"Total folders in PlantVillage: {len(all_folders)}")
print(f"Maize folders found: {len(maize_folders)}")
print("\nMaize classes:")
for folder in maize_folders:
    print(f"  - {folder}")

In [ ]:
dest_dir = '../data/plantvillage_maize_only'
os.makedirs(dest_dir, exist_ok=True)

for folder in maize_folders:
    src = os.path.join(source_dir, folder)
    dst = os.path.join(dest_dir, folder)
    if os.path.isdir(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f"Copied {folder} ({len(os.listdir(dst))} images)")

print(f"\nMaize-only dataset ready: {len(maize_folders)} classes")

## Step 3 — Load Data

In [ ]:
data_dir = '../data/plantvillage_maize_only'
IMG_SIZE = (224, 224)
BATCH_SIZE = 16  # reduce to 8 or 4 if memory issues

train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print(f"Training on {NUM_CLASSES} classes:")
for i, name in enumerate(class_names, 1):
    print(f"  {i}. {name}")

## Step 4 — Quick Data Check

In [ ]:
train_count = train_ds.cardinality().numpy() * BATCH_SIZE
val_count = val_ds.cardinality().numpy() * BATCH_SIZE
print(f"Training: ~{train_count} | Validation: ~{val_count}")

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout()
plt.show()

## Step 5 — Data Augmentation

In [ ]:
data_augmentation = Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Augmentation applied and datasets cached.")

## Step 6 — Build Model (ResNet50 Transfer Learning)

In [ ]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # freeze base during initial training

inputs  = Input(shape=(224, 224, 3))
x       = layers.Rescaling(1./255)(inputs)
x       = base_model(x, training=False)
x       = GlobalAveragePooling2D()(x)
x       = Dense(256, activation='relu')(x)
x       = Dropout(0.5)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 7 — Training Callbacks

In [ ]:
os.makedirs('../models', exist_ok=True)
os.makedirs('../logs', exist_ok=True)

callbacks = [
    ModelCheckpoint(
        filepath='../models/best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    CSVLogger('../logs/training_log.csv'),
    TensorBoard(log_dir='../logs', histogram_freq=1)
]

print("Callbacks ready.")
print(f"  Models  -> {os.path.abspath('../models')}")
print(f"  Logs    -> {os.path.abspath('../logs')}")

## Step 8 — Train
> Reduce `epochs` to 10 for a quick test run. Full training can take 1–6 hours depending on hardware.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

## Step 9 — Plot Training Curves
Reads from the saved CSV so this works even after reopening the notebook.

In [ ]:
history_df = pd.read_csv('../logs/training_log.csv')

print(f"Epochs completed: {len(history_df)}")
print(f"Final train accuracy : {history_df['accuracy'].iloc[-1]:.4f}")
print(f"Final val accuracy   : {history_df['val_accuracy'].iloc[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_df['accuracy'],     label='Train')
ax1.plot(history_df['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history_df['loss'],     label='Train')
ax2.plot(history_df['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 10 — Evaluate Model

In [ ]:
# Load model from disk if not already in memory
if 'model' not in dir():
    model = load_model('../models/best_model.h5')
    print("Model loaded from file.")

y_true, y_pred = [], []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

## Step 11 — Visualise Predictions

In [ ]:
def show_predictions(dataset, num_images=9):
    plt.figure(figsize=(15, 15))
    for images, labels in dataset.take(1):
        preds = model.predict(images)
        for i in range(min(num_images, len(images))):
            ax = plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype('uint8'))
            true_cls  = class_names[labels[i]]
            pred_cls  = class_names[np.argmax(preds[i])]
            conf      = np.max(preds[i]) * 100
            color     = 'green' if true_cls == pred_cls else 'red'
            plt.title(f"True: {true_cls}\nPred: {pred_cls} ({conf:.1f}%)", color=color)
            plt.axis('off')
    plt.tight_layout()
    plt.show()

show_predictions(val_ds)

## Step 12 — Save Model

In [ ]:
model.save('../models/maize_disease_model.h5')
print("Keras model saved.")

# TensorFlow Lite (mobile deployment)
converter   = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path  = '../models/maize_disease_model.tflite'

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved ({os.path.getsize(tflite_path) / 1e6:.2f} MB)")

# Training summary
history_df = pd.read_csv('../logs/training_log.csv')
summary = {
    'total_epochs':        int(len(history_df)),
    'final_train_accuracy': float(history_df['accuracy'].iloc[-1]),
    'final_val_accuracy':   float(history_df['val_accuracy'].iloc[-1]),
    'best_val_accuracy':    float(history_df['val_accuracy'].max()),
    'classes':             class_names
}

with open('../models/model_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)

print("Model summary saved to ../models/model_summary.json")

## Predict on a Single Image

In [ ]:
def predict_image(image_path):
    img       = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    preds           = model.predict(img_array)
    predicted_class = class_names[np.argmax(preds)]
    confidence      = np.max(preds) * 100

    print(f"Predicted : {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")
    return predicted_class, confidence

# predict_image('path/to/image.jpg')

## Resume After Closing Notebook
If the kernel was restarted, run Step 1 (imports), then this cell to reload data and model.

In [ ]:
data_dir = '../data/plantvillage_maize_only'

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(224, 224),
    batch_size=16
)

class_names = val_ds.class_names
model       = load_model('../models/best_model.h5')

print(f"Classes : {class_names}")
print("Model loaded — continue from Step 9 onwards.")